# CSV → pandas DataFrame

Читаем готовый результат BoardingData. Источник не изменяем, людей и поездки не сопоставляем. Подробные данные остаются локально.

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd

BASE = Path.cwd()
if not (BASE / "parse_csv.py").is_file():
    BASE = BASE / "parsers/CSV_Parser"
sys.path.insert(0, str(BASE))

In [2]:
from parse_csv import load_records

OUT = BASE / "result"
manifest = json.loads((OUT / "manifest.json").read_text())
{
    k: manifest[k]
    for k in ["tables", "record_statuses", "flight_date_min", "flight_date_max"]
}

{'tables': {'records': 155147, 'raw_records': 155147, 'issues': 55902}, 'record_statuses': {'parsed_with_uncertainty': 55902, 'parsed': 99245}, 'flight_date_min': '2017-01-01', 'flight_date_max': '2018-01-01'}

## Основная таблица

Выбираем несколько полей для просмотра. `load_records()` без columns загрузит все исходные и нормализованные поля. Номера документов, билетов и бронирований — строки.

In [3]:
df = load_records(
    OUT,
    columns=[
        "source_line_start",
        "flight_date",
        "flight_time",
        "flight_number",
        "destination_text",
        "baggage",
        "code_share",
        "passenger_birth_date_raw",
        "passenger_birth_date",
        "birth_year",
        "birth_date_candidates_json",
        "record_status",
    ],
)
print(df.shape)
df.head(8)

(155147, 12)


   source_line_start  ...            record_status
0                  2  ...  parsed_with_uncertainty
1                  3  ...                   parsed
2                  4  ...                   parsed
3                  5  ...                   parsed
4                  6  ...                   parsed
5                  7  ...                   parsed
6                  8  ...                   parsed
7                  9  ...                   parsed

[8 rows x 12 columns]

## Неоднозначность даты рождения

Дата 03/04/1980 допускает 4 марта и 3 апреля. Оба варианта сохранены, нормализованная дата отсутствует. Известный год не теряется. Доля принятых преобразований не является оценкой точности сопоставления.

In [4]:
df.loc[
    df.record_status == "parsed_with_uncertainty",
    [
        "passenger_birth_date_raw",
        "passenger_birth_date",
        "birth_date_candidates_json",
        "birth_year",
    ],
].head()

   passenger_birth_date_raw  ... birth_year
0                03/10/1983  ...       1983
9                07/04/1997  ...       1997
11               11/01/1971  ...       1971
15               04/05/2000  ...       2000
16               03/07/1990  ...       1990

[5 rows x 4 columns]

In [5]:
df.groupby(["baggage", "code_share"], dropna=False).size()

baggage     code_share
Delayed     Operated       2219
            Own           36355
None        Operated       2130
            Own           36802
Registered  Operated       2240
            Own           36717
Transit     Operated       2185
            Own           36499
dtype: int64

## Происхождение и сохранность

raw_records содержит точные исходные байты и декодированные поля. Номера строк — физические строки файла; запись с переносами внутри кавычек может занимать несколько строк.

In [6]:
raw = pd.read_parquet(
    OUT / "raw_records.parquet",
    columns=[
        "source_record_number",
        "source_line_start",
        "source_line_end",
        "record_status",
        "field_count",
    ],
)
raw.head()

   source_record_number  source_line_start  ...  record_status field_count
0                     1                  2  ...         parsed          14
1                     2                  3  ...         parsed          14
2                     3                  4  ...         parsed          14
3                     4                  5  ...         parsed          14
4                     5                  6  ...         parsed          14

[5 rows x 5 columns]

In [7]:
json.loads((OUT / "verification.json").read_text())

{'checks': {'source_sha256_matches': True, 'parquet_counts': True, 'all_source_bytes_reconstructed': True, 'raw_ids_unique_and_positions_contiguous': True, 'records_lineage': True, 'all_14_raw_fields_equal_independent_C_reader': True, 'identifier_types_and_leading_zeros': True, 'date32_types': True, 'all_flight_dates_equal_ISO_source': True, 'all_birth_candidates_and_unresolved_dates': True, 'baggage_None_category_retained': True, 'blank_records_separate': True}, 'records': 155147, 'raw_records': 155147, 'blank_records': 0, 'independent_reader': 'pandas read_csv engine=c; all raw values compared', 'limitation': 'Technical parsing verification; not passenger identity or actual-flight validation'}